# CustomerPulse — B2B SaaS Customer Health & Churn Risk Analytics

This notebook contains the analytical workflow behind the CustomerPulse Streamlit dashboard.

**Business goal:** help Customer Success teams identify at-risk accounts, quantify revenue exposure, understand the strongest churn signals, and prioritize intervention.

The current MVP uses transparent **rule-based risk logic** and an explainable **priority score**. Machine-learning churn prediction is planned as a later extension.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Expected project layout:
# CustomerPulse/
# ├── saas_accounts.csv
# └── notebooks/CustomerPulse.ipynb

data_path = Path("..") / "saas_accounts.csv"
df = pd.read_csv(data_path)

df.head()

## 1. Data quality and portfolio baseline

In [ ]:
print("Shape:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

total_accounts = len(df)
total_monthly_revenue = df["monthly_revenue"].sum()
churn_rate = df["churned_90d"].mean() * 100

pd.Series({
    "Total accounts": total_accounts,
    "Monthly revenue": total_monthly_revenue,
    "90-day churn rate (%)": churn_rate
})

### Baseline findings

- **50,000 accounts**
- **~$56.4M monthly revenue**
- **9.53% 90-day churn rate**
- No missing values or duplicate rows were found in the source dataset.


## 2. Churn signals — retained vs churned

In [ ]:
churn_comparison = df.groupby("churned_90d").agg(
    avg_seat_utilization=("seat_utilization", "mean"),
    avg_feature_adoption=("feature_adoption_pct", "mean"),
    avg_support_tickets=("support_tickets_30d", "mean"),
    avg_nps=("nps_score", "mean"),
    avg_csat=("csat_score", "mean")
).round(3)

churn_comparison.index = churn_comparison.index.map({0: "Retained", 1: "Churned"})
churn_comparison

### Interpretation

Churned accounts show a consistent deterioration in customer-health signals:

- lower seat utilization
- lower feature adoption
- more support tickets
- lower NPS
- lower CSAT

This made engagement weakness and support burden useful inputs for the first rule-based risk segment.


## 3. Rule-based risk populations

In [ ]:
risk_accounts = df.loc[
    (df["seat_utilization"] < 0.30) &
    (df["feature_adoption_pct"] < 30) &
    (df["support_tickets_30d"] >= 3)
].copy()

high_value_risk = df.loc[
    (df["churned_90d"] == 0) &
    (df["monthly_revenue"] > 1000) &
    (df["seat_utilization"] < 0.30) &
    (df["feature_adoption_pct"] < 30)
].copy()

high_value_watchlist = df.loc[
    (df["churned_90d"] == 0) &
    (df["monthly_revenue"] > 1000) &
    (
        (df["seat_utilization"] < 0.30) |
        (df["feature_adoption_pct"] < 30) |
        (df["support_tickets_30d"] >= 3)
    )
].copy()

print("Strict risk accounts:", len(risk_accounts))
print("High-value risk accounts:", len(high_value_risk))
print("High-value watchlist:", len(high_value_watchlist))

## 4. Revenue at risk — remove overlap before adding

In [ ]:
revenue_risk_index = risk_accounts.index.union(high_value_risk.index)

revenue_at_risk = df.loc[
    revenue_risk_index,
    "monthly_revenue"
].sum()

revenue_at_risk_pct = revenue_at_risk / total_monthly_revenue * 100

pd.Series({
    "Strict-risk revenue": risk_accounts["monthly_revenue"].sum(),
    "High-value-risk revenue": high_value_risk["monthly_revenue"].sum(),
    "Unique revenue at risk": revenue_at_risk,
    "Revenue at risk (%)": revenue_at_risk_pct
}).round(2)

The overlap is removed by using the union of account indexes. This avoids double-counting accounts that belong to both the strict-risk and high-value-risk populations.

**Unique monthly revenue at risk: ~$113.7K (~0.20% of portfolio monthly revenue).**

The low portfolio-level percentage should not be overstated: the strict rules intentionally identify a narrow population showing multiple warning signals.


## 5. Normalized risk segmentation

In [ ]:
risk_view = df.copy()
risk_view["risk_account"] = risk_view.index.isin(risk_accounts.index).astype(int)

def risk_rate_by(column):
    result = risk_view.groupby(column).agg(
        total_accounts=("risk_account", "size"),
        risk_accounts=("risk_account", "sum")
    )
    result["risk_rate_pct"] = (
        result["risk_accounts"] / result["total_accounts"] * 100
    )
    return result.sort_values("risk_rate_pct", ascending=False)

contract_risk = risk_rate_by("contract_type")
billing_risk = risk_rate_by("billing_plan")
region_risk = risk_rate_by("region")
industry_risk = risk_rate_by("industry")

contract_risk, billing_risk, region_risk, industry_risk

### Why normalize?

Raw risk-account counts can be misleading when one segment has far more customers than another. The normalized metric answers:

> **What percentage of this segment is currently classified as risky?**

Key MVP observations:
- monthly contracts have the highest normalized risk rate among contract types
- free-trial accounts have the highest normalized risk rate among billing plans
- APAC has the highest normalized regional risk rate, although regional differences are comparatively small


## 6. High-revenue risky accounts

In [ ]:
top_revenue_at_risk = risk_accounts[
    [
        "monthly_revenue",
        "contract_type",
        "billing_plan",
        "region",
        "industry",
        "seat_utilization",
        "feature_adoption_pct",
        "support_tickets_30d",
        "nps_score",
        "csat_score"
    ]
].sort_values("monthly_revenue", ascending=False).head(10)

top_revenue_at_risk

The highest-revenue risky accounts still show the same core engagement pattern: low seat utilization and low feature adoption. This is why CustomerPulse separates **risk detection** from **business prioritization**.


## 7. Explainable priority score

In [ ]:
def min_max_score(series, reverse=False):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        result = pd.Series(50.0, index=series.index)
    else:
        result = (series - minimum) / (maximum - minimum) * 100

    if reverse:
        result = 100 - result

    return result

risk_accounts["revenue_score"] = min_max_score(
    risk_accounts["monthly_revenue"]
)

risk_accounts["seat_weakness_score"] = min_max_score(
    risk_accounts["seat_utilization"],
    reverse=True
)

risk_accounts["feature_weakness_score"] = min_max_score(
    risk_accounts["feature_adoption_pct"],
    reverse=True
)

risk_accounts["support_score"] = min_max_score(
    risk_accounts["support_tickets_30d"]
)

nps_weakness = min_max_score(
    risk_accounts["nps_score"],
    reverse=True
)

csat_weakness = min_max_score(
    risk_accounts["csat_score"],
    reverse=True
)

risk_accounts["sentiment_score"] = (
    nps_weakness + csat_weakness
) / 2

risk_accounts["priority_score"] = (
    risk_accounts["revenue_score"] * 0.30 +
    risk_accounts["seat_weakness_score"] * 0.25 +
    risk_accounts["feature_weakness_score"] * 0.25 +
    risk_accounts["support_score"] * 0.15 +
    risk_accounts["sentiment_score"] * 0.05
).round(2)

### Priority-score weights

| Component | Weight |
|---|---:|
| Revenue exposure | 30% |
| Seat-utilization weakness | 25% |
| Feature-adoption weakness | 25% |
| Support burden | 15% |
| Sentiment weakness | 5% |

These are **MVP business weights**, not statistically optimized model coefficients. Their purpose is to create an interpretable Customer Success prioritization layer.


In [ ]:
priority_accounts = risk_accounts[
    [
        "billing_plan",
        "contract_type",
        "industry",
        "region",
        "monthly_revenue",
        "seat_utilization",
        "feature_adoption_pct",
        "support_tickets_30d",
        "nps_score",
        "csat_score",
        "priority_score"
    ]
].sort_values("priority_score", ascending=False)

priority_accounts.head(10)

## 8. Product takeaway

CustomerPulse is intentionally more than a churn summary.

The workflow separates:

1. **Portfolio health** — what is happening overall?
2. **Risk detection** — which accounts show multiple warning signals?
3. **Segmentation** — where is risk concentrated?
4. **Business prioritization** — which risky accounts matter most?
5. **Customer Success action** — which accounts should be investigated first?

### Next iteration

- add risk-reason labels so CS teams can filter by the behavior driving risk
- add account-level drill-down explaining why an account is prioritized
- add intervention recommendations by risk reason
- compare the rule-based system with a supervised churn-classification model
- only add trend analysis when true historical account-level data is available
